In [1]:
from __future__ import annotations

In [2]:
import torch
import tiktoken

import torch.nn as nn
import torch.nn.functional as F

from typing import Tuple, Optional, List

In [4]:

data_path = 'shakespeare.txt'
with open(data_path, 'r', encoding='utf-8') as f:
    data = f.read()
    
data[:1000]

"\ufeff1603\n\nALLS WELL THAT ENDS WELL\n\nby William Shakespeare\n\nDramatis Personae\n\n  KING OF FRANCE\n  THE DUKE OF FLORENCE\n  BERTRAM, Count of Rousillon\n  LAFEU, an old lord\n  PAROLLES, a follower of Bertram\n  TWO FRENCH LORDS, serving with Bertram\n\n  STEWARD, Servant to the Countess of Rousillon\n  LAVACHE, a clown and Servant to the Countess of Rousillon\n  A PAGE, Servant to the Countess of Rousillon\n\n  COUNTESS OF ROUSILLON, mother to Bertram\n  HELENA, a gentlewoman protected by the Countess\n  A WIDOW OF FLORENCE.\n  DIANA, daughter to the Widow\n\n  VIOLENTA, neighbour and friend to the Widow\n  MARIANA, neighbour and friend to the Widow\n\n  Lords, Officers, Soldiers, etc., French and Florentine  \n\nSCENE:\nRousillon; Paris; Florence; Marseilles\n\nACT I. SCENE 1.\nRousillon. The COUNT'S palace\n\nEnter BERTRAM, the COUNTESS OF ROUSILLON, HELENA, and LAFEU, all in black\n\n  COUNTESS. In delivering my son from me, I bury a second husband.\n  BERTRAM. And I in g

In [204]:
from typing import Any

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class MultiLayerPerceptron(nn.Module):
    def __init__(self,embed_dim:int, bias: bool, drop_out: float=0.2) -> None:
        super().__init__()
        
        self.c_fc = nn.Linear(embed_dim, embed_dim * 4, bias=bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(embed_dim *4, embed_dim, bias=bias)
        self.dropout = nn.Dropout(drop_out)
        
    def forward(self, X: torch.Tensor)-> torch.Tensor:
        X = self.c_fc(X)
        X = self.gelu(X)
        X = self.c_proj(X)
        X = self.dropout(X)
        return X

class MultiHeadAttention(nn.Module):
    """
    MultiHeadAttention that runs several different heads in parallel,
    each head learning a different relationships from the other heads
    """
    def __init__(self, num_heads: int, embed_dim: int, block_size: int, bias: bool, drop_out: float):
        """
        Initialize multi-head attention.

        Set up linear projections and validate configuration

        Args:
            embed_dim (int): Embedding dimension of the input and output tensors
            n_heads (int): Number of parallel heads to run in the attention mechanism
        """
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"
        self.num_heads: int = num_heads
        self.embed_dim: int = embed_dim
        self.head_dim : int = embed_dim // num_heads
        
        #* query, key and values projections
        self.c_att = nn.Linear(embed_dim , embed_dim * 3, bias = False)
        
        #* output projection
        self.c_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        
        #* attention drop out and out put dropout
        self.att_dropout = nn.Dropout(drop_out)
        self.res_dropout = nn.Dropout(drop_out)
        
        #* flash attention
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                        .view(1, 1, block_size, block_size))
        
        
    def forward(self, embedding: torch.Tensor) -> torch.Tensor:
        q,k,v = self.c_att(embedding).split(self.embed_dim, dim=2)
        
        B, T, C = q.shape       #* shape (batch, seq_len, embed_dim)
        #* split the heads      #* shape (batch, num_heads, seq_len, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        
        #* automatic self attention
        if self.flash:
            #* y.shape -> B, num_heads, T,T
            y = F.scaled_dot_product_attention(q, k, v, attn_mask= None, is_causal=True)
        else:
            #* manual implementation
            att = (q @ k.transpose(-1,-2)) / (self.embed_dim ** 0.5)
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = torch.softmax(att, dim= -1, dtype = torch.long)
            att = self.att_dropout(att)
            y = att @ v
            
        #* merge all the heads back together
        y = y.transpose(1,2).contiguous().view(B,T,C)
        
        #* output projection
        y = self.res_dropout(self.c_proj(y))
    
        return y
    
class Block(nn.Module):
    def __init__(self,num_heads: int, embed_dim:int, block_size: int, bias: bool=False, drop_out:float = 0.2) -> None:
        super().__init__()
        self.mha = MultiHeadAttention(num_heads, embed_dim, block_size, bias, drop_out)
        self.mlp = MultiLayerPerceptron(embed_dim, bias=bias, drop_out= drop_out)
        self.norm1 = LayerNorm(embed_dim, bias=bias)
        self.norm2 = LayerNorm(embed_dim, bias=bias)
        
    def forward(self, X: torch.Tensor)-> torch.Tensor:
        X = X + self.mha(self.norm1(X))
        X = X + self.mlp(self.norm2(X))
        return X

class Transformer(nn.Module):
    def __init__(self,
                vocab_size: int, 
                embed_dim: int,
                num_heads: int, 
                block_size:int,
                num_blocks: int,
                bias: bool= False,
                drop_out: float = 0.2)-> None:
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim) #* shape(vocab_size, embed_dim)
        self.posit_embed = nn.Embedding(vocab_size, embed_dim)
        self.embed_drop_out = nn.Dropout(drop_out)
        self.transformer_blocks = [
            Block(num_heads, embed_dim, block_size, bias, drop_out) for _ in range(num_blocks)]
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias = False)
        
        #* weight tying
        self.token_embed.weight = self.lm_head.weight
        
        #* init all weights
        self.apply(self._init_weights)
        #* apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/(2 * num_blocks) ** 0.5)

        #* report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))
        
    def get_num_params(self, non_embedding: bool = True) -> int:
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.token_embed.weight.numel()
        return n_params
    
    def forward(self, 
                idx: torch.Tensor, 
                targets: Optional[torch.Tensor] =  None
                ) -> Tuple[torch.Tensor, torch.Tensor | None]:
        
        _,t = idx.size()
        positions = torch.arange(0,t, dtype= torch.long)
        
        token_embed = self.token_embed(idx)
        pos_embed = self.posit_embed(positions)
        x = self.embed_drop_out(token_embed + pos_embed)
        
        for block in self.transformer_blocks:
            x = block(x)
        
        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None
        
        return logits, loss
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_length: int, ) -> torch.Tensor:
        for _ in range(max_length):
            out, _= self(idx, targets=None)
            out = out[:, -1, :]
            out = F.softmax(out, dim=-1)
            idx_next = torch.multinomial(out, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
    

In [165]:

tokenizer = tiktoken.get_encoding("gpt2")
# idx = torch.tensor(tokenizer.encode(data), dtype=torch.long)

In [206]:
########################################################################################
#* Hyperparameters.
#* ======================================================================================
vocab_size = tokenizer.n_vocab
embed_dim = 16
block_size = 64
batch_size = 32
number_epochs = 500
lr = 3e-3
eval_iters = 1
num_heads = 4
num_blocks = 4
drop_out = 0.2
bias = False

#* =======================================================================================

#* instantiate the model and optimizer objects
model = Transformer(vocab_size, embed_dim, num_heads,block_size,num_blocks,bias,drop_out)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)

#* =======================================================================================

#* get the train and val data
n = 0.9
train_data = idx[:int(n*len(idx))]
val_data = idx[int(n*len(idx)):]

#* batch the data
def get_batch(split: str = 'train')->Tuple[torch.Tensor, torch.Tensor]:
    data = train_data if split == 'train' else val_data
    idx = torch.randint(len(data) - block_size, size=(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idx])
    y = torch.stack([data[i+1:i+block_size+1] for i in idx])
    return x,y

#* ========================================================================================

#* model evaluation
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, targets=y)
            losses[k] = loss.item()
        out[split] = losses.mean()
        
    model.train()
    return {
        'train_loss': out['train'],
        'val_loss': out['val']
    }

#* ==================================================================================

#* model training
for iter in range(number_epochs):
    #* evaluate loss at intervals
    if iter % eval_iters == 0:
        losses = estimate_loss()
        train_loss, val_loss = losses['train_loss'], losses['val_loss']
        print(f'Epoch {iter}: train_loss {train_loss}: val_loss {val_loss}')
        
    #* forward pass of the model
    x,y = get_batch('train')
    logits, loss = model(x,targets=y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
#* ==================================================================================

number of parameters: 0.80M
Epoch 0: train_loss 10.825813293457031: val_loss 10.820945739746094
Epoch 1: train_loss 10.808186531066895: val_loss 10.81352424621582
Epoch 2: train_loss 10.800029754638672: val_loss 10.814083099365234
Epoch 3: train_loss 10.762327194213867: val_loss 10.802376747131348
Epoch 4: train_loss 10.71070671081543: val_loss 10.771588325500488
Epoch 5: train_loss 10.667827606201172: val_loss 10.73515510559082
Epoch 6: train_loss 10.585655212402344: val_loss 10.687508583068848
Epoch 7: train_loss 10.557680130004883: val_loss 10.635682106018066
Epoch 8: train_loss 10.505696296691895: val_loss 10.58487606048584
Epoch 9: train_loss 10.442317962646484: val_loss 10.527633666992188
Epoch 10: train_loss 10.330989837646484: val_loss 10.476420402526855
Epoch 11: train_loss 10.267229080200195: val_loss 10.413796424865723
Epoch 12: train_loss 10.233414649963379: val_loss 10.364681243896484
Epoch 13: train_loss 10.114720344543457: val_loss 10.302468299865723
Epoch 14: train_loss

In [200]:
context = torch.ones((1,1) ,dtype =torch.long)
next_idx = model.generate(context, max_length=10).tolist()
generate_text = tokenizer.decode(next_idx[0])
print(generate_text)

"evidenceenting OF
 this De Cool poolquest Or


In [196]:
from __future__ import annotations

import numpy as np
import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPT2Config:
    vocab_size: int = 50257
    seq_length: int = 1024
    embed_dim: int = 768
    num_layers: int = 12
    num_heads: int = 12
    bias: bool = True
    drop_out: float = 0.2
    

class MultiHeadAttention(nn.Module):
    """
    MultiHeadAttention that runs several different heads in parallel,
    each head learning a different relationships from the other heads
    """
    def __init__(self, config):
        """
        Initialize multi-head attention.

        Set up linear projections and validate configuration

        Args:
            embed_dim (int): Embedding dimension of the input and output tensors
            n_heads (int): Number of parallel heads to run in the attention mechanism
        """
        super().__init__()
        self.config = config
        assert config.embed_dim % config.num_heads == 0, "Embedding dimension must be divisible by number of heads"
        self.num_heads: int = config.num_heads
        self.embed_dim: int = config.embed_dim
        self.head_dim : int = config.embed_dim // config.num_heads
        
        #* query, key and values projections
        self.c_attn = nn.Linear(config.embed_dim , config.embed_dim * 3, bias = config.bias)
        
        #* output projection
        self.c_proj = nn.Linear(config.embed_dim, config.embed_dim, bias=config.bias)
        
        #* attention drop out and out put dropout
        self.attn_dropout = nn.Dropout(config.drop_out)
        self.res_dropout = nn.Dropout(config.drop_out)
        
        #* flash attention
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))
        
        
    def forward(self, embedding: torch.Tensor) -> torch.Tensor:
        q,k,v = self.c_attn(embedding).split(self.embed_dim, dim=2)
        
        B, T, C = q.shape       #* shape (batch, seq_len, embed_dim)
        #* split the heads      #* shape (batch, num_heads, seq_len, head_dim)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        
        #* automatic self attention
        if self.flash:
            #* y.shape -> B, num_heads, T,T
            y = F.scaled_dot_product_attention(q, k, v, attn_mask= None, is_causal=True)
        else:
            #* manual implementation
            att = (q @ k.transpose(-1,-2)) / (self.embed_dim ** 0.5)
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = torch.softmax(att, dim= -1, dtype = torch.long)
            att = self.attn_dropout(att)
            y = att @ v
            
        #* merge all the heads back together
        y = y.transpose(1,2).contiguous().view(B,T,C)
        
        #* output projection
        y = self.res_dropout(self.c_proj(y))
    
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.embed_dim, 4* config.embed_dim, bias =config.bias)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.embed_dim, config.embed_dim, bias= config.bias)
    
    def forward(self, x:torch.Tensor)->torch.Tensor:
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self,config) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.embed_dim, bias=config.bias)
        self.attn = MultiHeadAttention(config)
        self.ln_2 = nn.LayerNorm(config.embed_dim, bias=config.bias)
        self.mlp = MLP(config)
        
    def forward(self, X: torch.Tensor)-> torch.Tensor:
        X = X + self.attn(self.ln_1(X))
        X = X + self.mlp(self.ln_2(X))
        return X

class GPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.embed_dim),
            wpe = nn.Embedding(config.seq_length, config.embed_dim),
            h = nn.ModuleList([Block(config)
                            for _ in range(config.num_layers)]),
            ln_f = nn.LayerNorm(config.embed_dim),
        ))
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)
        
    def forward(self, idx: torch.Tensor,) -> torch.Tensor:
        _,t = idx.size()
        positions = torch.arange(0,t, dtype= torch.long)
        
        token_embed = self.transformer.wte(idx)
        pos_embed = self.transformer.wpe(positions)
        x = token_embed + pos_embed
        
        for layer in self.transformer.h:
            x = layer(x)
            
        x = self.transformer.ln_f(x) 
        logits = self.lm_head(x)
        
        return logits
    
    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_length: int,topk: int ) -> torch.Tensor:
        for _ in range(max_length):
            out = self(idx)
            out = out[:, -1, :]
            out = F.softmax(out, dim=-1)
            #* topk sampling
            top_probs, top_indices = torch.topk(out, k =topk, dim=-1)
            idx_next = torch.multinomial(top_probs.type(torch.float32), num_samples=1)
            xcol = torch.gather(top_indices, -1, idx_next)
            idx = torch.cat((idx, xcol), dim=1)
        return idx
        
    @classmethod
    def from_pretrained(cls, model_name: str ='gpt2') -> GPT2:
        print('⏳ Loading the pretrained weights from hugging face...')
        assert model_name in ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl']
        from transformers import GPT2LMHeadModel
        #* get the model configuration
        config = {
            'gpt2': dict(embed_dim= 768, num_layers= 12, num_heads= 12),
            'gpt2-medium': dict(embed_dim= 1024, num_layers= 24, num_heads= 16),
            'gpt2-large': dict(embed_dim= 1280, num_layers= 36, num_heads= 20),
            'gpt2-xl': dict(embed_dim= 1600, num_layers= 48, num_heads= 25)
        }[model_name]
        
        config['vocab_size'] = 50257
        config['seq_length'] = 1024
        
        #* instantiate the model: both our model and hf model
        #* and get their state dictionaries
        config = GPT2Config(**config)
        model = GPT2(config)
        sd = model.state_dict()
        model_keys = sd.keys()
        model_keys = [key for key in model_keys if not key.endswith('.attn_bias')]
        
        hf_model = GPT2LMHeadModel.from_pretrained('gpt2')
        hf_sd = hf_model.state_dict()
        hf_keys = hf_sd.keys()
        hf_keys = [key for key in hf_keys if not key.endswith('.attn_masked_bias')]
        hf_keys = [key for key in hf_keys if not key.endswith('.attn_bias')]
        
        assert len(model_keys) == len(hf_keys), f'mismatched keys, expected {len(hf_keys)}, got {len(model_keys)}'
        #* transpose hugging face weights to match model weight shape
        transpose = ['c_attn.weight', 'c_proj.weight', 'c_fc.weight', 'c_proj.weight']
        
        for key in model_keys:
            if any(key.endswith(w) for w in transpose):
                if key not in hf_keys:
                    raise KeyError(
                        f"Key '{key}' not found in hugging face state dictionary"
                    )
                assert hf_sd[key].shape[::-1] == sd[key].shape, f'mismatched shapes, expected {sd[key].shape}, got {hf_sd[key].shape[::1]}'
                with torch.no_grad():
                    sd[key].copy_(hf_sd[key].T)
            else:
                assert hf_sd[key].shape == sd[key].shape
                with torch.no_grad():
                    sd[key].copy_(hf_sd[key])
                
        print('✅ Loaded pretrained weights from hugging face')
        print("🤗 Yaah, we didn't crush")
        return model

In [199]:
num_return_sequences = 10
model = GPT2.from_pretrained()

torch.manual_seed(42)
tokens = torch.tensor(tokenizer.encode('Hi, I am a Large Language Model'), dtype= torch.long)
tokens = tokens.unsqueeze(0)
next_tokens = model.generate(tokens, max_length= 50, topk=20)
generated_text = tokenizer.decode(next_tokens.tolist()[0])
print(f'Generated Text: ', generated_text)

⏳ Loading the pretrained weights from hugging face...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1958.71it/s]


✅ Loaded pretrained weights from hugging face
🤗 Yaah, we didn't crush
Generated Text:  Hi, I am a Large Language Model, but I do not know how I should make a list. If you want to know how I make this, I will explain it below or see how to put it into another book:

My goal is to have a list of the most


In [189]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', 'gpt2')
set_seed(42)
generator('Hi, am a Large Language Model ', max_length=30, num_return_sequences=5)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1979.39it/s]
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hi, am a Large Language Modeler. I want to make programming languages more efficient and flexible.\n\nI am a Small Language Modeler. I want to make programming languages more efficient and flexible.\n\nI am a Beginner Language Modeler. I want to make programming languages more efficient and flexible.\n\nI am a Software Developer. I want to make programming languages more efficient and flexible.\n\nI am a Professional Language Modeler. I want to make programming languages more efficient and flexible.'},
 {'generated_text': "Hi, am a Large Language Modeler in Google and I'm doing a lot of research on language models. I've seen a lot of good things, but I don't really think it's good enough.\n\nI'm sure my questions are answered, but I haven't really thought about them since I started my book.\n\nI'm going to start with the big picture.\n\nWhat is the biggest difference between you and any of the other people who have been working on this book?\n\nI think that the big